In [1]:
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept="*", multiple=True)
display(uploader)

FileUpload(value=(), accept='*', description='Upload', multiple=True)

In [6]:
import pandas as pd

filename = "housing_residential_processed.csv.gz"

df = pd.read_csv(filename, compression="gzip")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1353121 entries, 0 to 1353120
Data columns (total 15 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   address                 1353121 non-null  object 
 1   longitude               1353121 non-null  float64
 2   latitude                1353121 non-null  float64
 3   area                    1353121 non-null  float64
 4   room_count              1206627 non-null  float64
 5   floor                   1353061 non-null  float64
 6   floor_count             142121 non-null   float64
 7   market_type             1353121 non-null  object 
 8   flat_type               1353121 non-null  object 
 9   ceiling_height          47225 non-null    float64
 10  build_year              1245309 non-null  float64
 11  balcony                 1353121 non-null  bool   
 12  price                   1353121 non-null  float64
 13  price_per_square_meter  1353121 non-null  float64
 14  da

Вот тут я типы поменяла на нужные нам

In [4]:
df = df.astype(
    {
        "room_count": "Int64",
        "floor": "Int64",
        "build_year": "Int64",
        "date": "datetime64[ns]",
    }
)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1353121 entries, 0 to 1353120
Data columns (total 15 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   address                 1353121 non-null  object        
 1   longitude               1353121 non-null  float64       
 2   latitude                1353121 non-null  float64       
 3   area                    1353121 non-null  float64       
 4   room_count              1206627 non-null  Int64         
 5   floor                   1353061 non-null  Int64         
 6   floor_count             142121 non-null   float64       
 7   market_type             1353121 non-null  object        
 8   flat_type               1353121 non-null  object        
 9   ceiling_height          47225 non-null    float64       
 10  build_year              1245309 non-null  Int64         
 11  balcony                 1353121 non-null  bool          
 12  price         

In [8]:
df.describe()

,longitude,latitude,area,room_count,floor,floor_count,ceiling_height,build_year,price,price_per_square_meter
count,1.353121e+06,1.353121e+06,1.353121e+06,1.206627e+06,1.353061e+06,142121.000000,47225.000000,1.245309e+06,1.353121e+06,1.353121e+06
mean,3.757826e+01,5.571673e+01,7.802964e+01,1.707991e+00,1.022691e+01,13.801465,3.398022,2.020174e+03,1.615010e+07,2.138888e+05
std,2.048560e-01,1.303422e-01,1.338137e+04,8.220080e-01,7.362132e+00,7.975981,48.910175,1.597943e+01,3.061419e+09,3.377936e+05
min,3.690146e+01,5.515348e+01,1.000000e-01,0.000000e+00,-1.100000e+01,0.000000,0.000000,0.000000e+00,8.905000e+03,2.653000e+02
25%,3.743264e+01,5.561525e+01,3.530000e+01,1.000000e+00,5.000000e+00,8.000000,2.500000,2.019000e+03,5.395500e+06,1.234330e+05
50%,3.757200e+01,5.571440e+01,4.477000e+01,2.000000e+00,9.000000e+00,14.000000,2.600000,2.021000e+03,8.000000e+06,1.827000e+05
75%,3.772550e+01,5.581216e+01,6.136000e+01,2.000000e+00,1.400000e+01,17.000000,2.700000,2.023000e+03,1.260000e+07,2.607530e+05
max,3.804808e+01,5.604984e+01,1.000000e+07,2.900000e+01,9.990000e+02,126.000000,4150.000000,2.473000e+03,3.047454e+12,2.977186e+08


Дальше хочу посмотреть на все значения которые не "физичны" для нашей ситуации:

### 1. Площадь: у нас сейчас минимум это 1 кв.м а максимум 1.0000e+07 кв.м. Что бы понять границы надо смотреть какие вообще реально исторические минимум и максимумы у нас есть на рынке недвижимости в Москве.
 - Абсолютный минимум на рынке Подмосковья встречались студии площадью от 17,5 квадратных метров.
 - Абсолютный максимум по жилой квартире в Мосве и подмосковье был зафиксирован как 1100 кв.м

Посмотрим в наш дата-сет:

In [9]:
area_error = df[(df["area"] <= 17.5) | (df["area"] > 1100)]
area_error

,address,longitude,latitude,area,room_count,floor,floor_count,market_type,flat_type,ceiling_height,build_year,balcony,price,price_per_square_meter,date
822,"область Люберцы Октябрьский проспект, д. 403",37.298509,55.477516,16.4,0.0,3.0,4.0,secondary,studio,NaN,NaN,False,3.731000e+06,227500.00,2025-05-01
864,"проезд, д. 1, стр. 3",37.267077,55.430130,15.3,0.0,3.0,3.0,secondary,studio,NaN,NaN,False,3.213000e+06,210000.00,2025-06-10
940,"Зеленая, д. 1",37.997982,55.522052,17.0,0.0,2.0,3.0,secondary,studio,NaN,NaN,False,3.500000e+06,205882.40,2025-05-13
1000,"проезд, д. 1",37.267077,55.430130,15.8,0.0,2.0,3.0,secondary,studio,NaN,NaN,False,3.555000e+06,225000.00,2025-05-21
1363,"Российская, д. 40",37.762533,55.669224,17.5,0.0,5.0,6.0,secondary,studio,NaN,NaN,False,2.500000e+06,142857.10,2025-05-13
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1341966,"Марушкино дер., к. 1 (3 оч.)",37.200386,55.595394,6316.0,2.0,3.0,NaN,primary,flat,NaN,2025.0,False,8.507652e+08,134700.00,2025-01-31
1342209,"Лесные Поляны 5-я ул., к. 2.2",37.462935,55.583722,14.1,NaN,3.0,NaN,primary,flat,NaN,2025.0,False,6.264992e+06,444325.67,2025-01-31
1343485,"Лесные Поляны 5-я ул., к. 2.2",37.462935,55.583722,14.1,NaN,4.0,NaN,primary,flat,NaN,2025.0,False,6.301780e+06,446934.75,2025-02-17
1345162,"Лесные Поляны 5-я ул., к. 2.2",37.462935,55.583722,14.1,NaN,6.0,NaN,primary,flat,NaN,2025.0,False,5.764465e+06,408827.30,2025-03-11


**Вывод:**
обрезаем все что меньше 17.5 и все что больше 1100

### 2. Дальше смотрим по кочеству этажей в доме:
- Также самый низкий этаж в москве это -4, а самый высокий это 85, у нас же в дате-сете вот такие интересности

In [5]:
floor_error = df[(df["floor"] <= -4) | (df["floor"] > 85)]
floor_error

,address,longitude,latitude,area,room_count,floor,floor_count,market_type,flat_type,ceiling_height,build_year,balcony,price,price_per_square_meter,date
91578,"Новомытищинский проспект, д. 43, к. 4",37.727975,55.900854,88.00,3,99,8.0,secondary,flat,NaN,<NA>,False,14900000.00,169318.2,2025-03-09
92467,Новомытищинский проспект,37.736994,55.910841,88.00,3,99,8.0,secondary,flat,NaN,<NA>,False,14900000.00,169318.2,2025-05-09
103985,"Суздальский проспект, nan",37.617644,55.755819,52.30,1,225,15.0,secondary,flat,2.7,2026,False,14000000.00,267686.4,2024-12-15
134095,"nan, 11",37.617644,55.755819,999.00,4,999,2.0,secondary,flat,NaN,<NA>,False,10000000.00,10010.0,2024-10-17
181019,"Рублевское ш., вл. 68-70, к. 2",37.376402,55.770780,47.70,1,-9,NaN,primary,flat,NaN,2017,False,7284553.20,152716.0,2017-03-09
692953,"Саввино мкр., к. 2",38.027743,55.729797,42.26,1,259,NaN,primary,flat,NaN,2017,False,2794527.02,66127.0,2017-06-26
735974,"вблизи г. Красногорска, д. 2",37.246155,55.817026,35.50,1,-11,NaN,primary,flat,NaN,2018,False,4447191.50,125273.0,2018-05-14
766751,"Климовск мкр., Серпуховская ул., д. 7",37.541467,55.353606,42.80,1,121,NaN,primary,flat,NaN,2018,False,2340518.00,54685.0,2018-11-07
1145315,"Внуковское п., Ликова дер., к. 1",37.278117,55.621287,35.21,1,211,NaN,primary,flat,NaN,2016,False,3671804.43,104283.0,2016-03-11
1168504,"Сосенское п., вблизи д. Столбово, уч. 2, д. 6",37.466476,55.555382,54.52,2,466,NaN,primary,flat,NaN,2017,False,5520858.76,101263.0,2017-11-13


Ну тут не сильно много поэтому смело кикаем если этаж меньше -4 или больше 85

### 3. Цена. 
Посмотрела на минимумы и максимумы за 2004-2025 года, получила вот такие циферки:
Минимальная цена 1,1 млн рублей (эквивалент 2004 года). А ценовой потолок для элитного жилья взлетел до фантастических 7,3 млрд рублей

In [16]:
price_error = df[(df["price"] < 1_100_000) | (df["price"] > 7_300_000_000)]
price_error

,address,longitude,latitude,area,room_count,floor,floor_count,market_type,flat_type,ceiling_height,build_year,balcony,price,price_per_square_meter,date
4529,"Фабричная, д. 1",37.390856,55.852693,18.50,1.0,1.0,1.0,secondary,flat,NaN,NaN,False,1.050000e+06,56756.80,2025-05-13
25624,"7-я Ильича, д. 6",37.617644,55.755819,28.00,1.0,2.0,2.0,secondary,flat,NaN,NaN,False,1.050000e+06,37500.00,2025-06-10
26401,"Мира, д. 10",37.632682,55.773879,32.10,0.0,1.0,4.0,secondary,studio,NaN,NaN,False,1.000000e+06,31152.60,2025-06-10
29387,"Юбилейная, д. 3",37.565497,55.508136,33.40,0.0,3.0,3.0,secondary,studio,NaN,NaN,False,1.030620e+06,30856.90,2025-06-10
35578,"Мира, д. 19А",37.632134,55.778031,25.70,1.0,2.0,3.0,secondary,flat,NaN,NaN,False,1.050000e+06,40856.00,2025-03-14
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1251985,"Рязановское п., к. 10",37.514496,55.495044,1.64,2.0,3.0,NaN,primary,flat,NaN,2021.0,False,2.792067e+05,170247.99,2021-06-15
1254031,"вблизи Алхимово дер., д. 3.1",37.527174,55.476572,3.60,NaN,2.0,NaN,primary,flat,NaN,2021.0,False,7.970436e+05,221401.00,2021-07-12
1263932,"Московский п., в районе дер. Саларьево, к. 54",37.411195,55.608110,3.50,1.0,5.0,NaN,primary,flat,NaN,2021.0,False,7.525000e+05,215000.00,2021-12-15
1286942,"Коммунарка пос., д. 14, к. 2",37.503466,55.554554,21.00,1.0,4.0,NaN,primary,flat,NaN,2023.0,False,6.290172e+05,29953.20,2023-03-25


По цене не понимаю пока на сколько это правильно так делать, но впринципе 151 строку можем и выкинуть, кажется что тут есть значительно пересечение с тем что мы выкинем по площади поэтому вообще норм получается